In [1]:
%cd ..

c:\Users\mjane\Documents\GitHub\LSTM-FAISS-DTW


In [2]:
import numpy as np
import os
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
from Recognition.faiss_dtw import build_faiss_index, load_faiss_index, benchmark_faiss, benchmark_faiss_dtw
from Models.LSTM_encoder import LSTMEncoder

In [3]:
model_name = LSTMEncoder.__name__

train = np.load(f"Models/{model_name}_Checkpoints/train_embeddings.npz",allow_pickle=True)
val = np.load(f"Models/{model_name}_Checkpoints/val_embeddings.npz",allow_pickle=True)

In [4]:
train_emb = train["last_embeddings"]
train_seq = train["seq_embeddings"]
train_labels = train["labels"]

val_emb = val["last_embeddings"]
val_seq = val["seq_embeddings"]
val_labels = val["labels"]

In [5]:
knn = KNeighborsClassifier(
        n_neighbors=1,
        metric="euclidean"
    )

knn.fit(train_emb, train_labels)
predictions = knn.predict(val_emb)
model_acc = accuracy_score(val_labels, predictions)
model_acc *= 100

In [6]:
build_faiss_index(train_emb)
index = load_faiss_index()

In [7]:
faiss_acc = benchmark_faiss(index, val_emb, val_labels, train_labels)

In [8]:
dtw_acc = benchmark_faiss_dtw(index, val_emb, val_seq, val_labels, train_seq, train_labels, k=10)

In [9]:
print(f"{model_name} accuracy           : {model_acc:.2f} %")
print(f"FAISS accuracy                  : {faiss_acc:.2f} %")
print(f"FAISS + DTW accuracy            : {dtw_acc:.2f} %")

LSTMEncoder accuracy           : 85.37 %
FAISS accuracy                  : 85.37 %
FAISS + DTW accuracy            : 85.07 %


In [10]:
val.close()
train.close()